In [3]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

AI_DATA = "csv/ts/cleaned_dataset_AI.csv"
NO_AI_DATA = "csv/ts/cleaned_dataset_nonAI.csv"

In [ ]:
# Average of lines changed
df_AI = pd.read_csv(AI_DATA)
df_NO_AI = pd.read_csv(NO_AI_DATA)

mean_value_AI = df_AI["lines"].mean()
mean_value_NO_AI = df_NO_AI["lines"].mean()

print(f"Mean_value of AI for average lines changed: {mean_value_AI}")
print(f"Mean_value of NON AI for average lines changed: {mean_value_NO_AI}")


data = [
    df_AI["lines"].dropna(),
    df_NO_AI["lines"].dropna()
]

plt.violinplot(data, showmeans=True)

plt.xticks([1,2], ["AI", "Non-AI"])
plt.ylabel("Insertion / Deletion Ratio")
plt.title("Distribution of Insertion/Deletion Ratio")

plt.show()




In [ ]:
df_AI = pd.read_csv(AI_DATA)
df_NO_AI = pd.read_csv(NO_AI_DATA)

df_AI["ratio_insertion_deletion"] = df_AI["insertions"] / df_AI["deletions"].replace(0, np.nan)
df_NO_AI["ratio_insertion_deletion"] = df_NO_AI["insertions"] / df_NO_AI["deletions"].replace(0, np.nan)

ratio_value_AI = df_AI["ratio_insertion_deletion"].mean()
ratio_NO_AI = df_NO_AI["ratio_insertion_deletion"].mean()

print(f"Ratio of insertion/deletion for AI: {ratio_value_AI}")
print(f"Ratio of insertion/deletion for NON AI: {ratio_NO_AI}")



data = [
    df_AI["ratio_insertion_deletion"].dropna(),
    df_NO_AI["ratio_insertion_deletion"].dropna()
]

plt.violinplot(data, showmeans=True)

plt.xticks([1,2], ["AI", "Non-AI"])
plt.ylabel("Insertion / Deletion Ratio")
plt.title("Distribution of Insertion/Deletion Ratio")

plt.show()

In [ ]:
df_AI = pd.read_csv(AI_DATA)
df_NO_AI = pd.read_csv(NO_AI_DATA)



mean_insertions_AI = df_AI["insertions"].mean()
mean_insertions_NO_AI = df_NO_AI["insertions"].mean()

print(f" Averaeg for insertions for AI: {mean_insertions_AI}")
print(f"Average of insertions for NON AI: {mean_insertions_NO_AI}")



data = [
    df_AI["insertions"].dropna(),
    df_NO_AI["insertions"].dropna()
]

plt.violinplot(data, showmeans=True)

plt.xticks([1,2], ["AI", "Non-AI"])
plt.ylabel("Insertions")
plt.title("Distribution of inserions")

plt.show()

In [ ]:
df_AI = pd.read_csv(AI_DATA)
df_NO_AI = pd.read_csv(NO_AI_DATA)



mean_insertions_AI = df_AI["deletions"].mean()
mean_insertions_NO_AI = df_NO_AI["deletions"].mean()

print(f"Average for deletions for AI: {mean_insertions_AI}")
print(f"Average of deletions for NON AI: {mean_insertions_NO_AI}")



data = [
    df_AI["deletions"].dropna(),
    df_NO_AI["deletions"].dropna()
]

plt.violinplot(data, showmeans=True)

plt.xticks([1,2], ["AI", "Non-AI"])
plt.ylabel("Deletions")
plt.title("Distribution of deletions")

plt.show()

In [ ]:
from pydriller import Repository

df_AI = pd.read_csv(AI_DATA)
df_NO_AI = pd.read_csv(NO_AI_DATA)

# change df_AI to df_NO_AI to save non-ai
hashes_set = set(df_NO_AI["hash"])
reverted_commits = {}

for commit in Repository("https://github.com/pancakeswap/pancake-frontend").traverse_commits():
    if "reverts commit" in (commit.msg or "").lower():
        # extract the reverted hash
        import re
        match = re.search(r"reverts commit (\w+)", commit.msg, re.IGNORECASE)
        if match:
            reverted_hash = match.group(1)
            if reverted_hash in hashes_set:
                reverted_commits[reverted_hash] = commit.hash

# convert to dataframe
import pandas as pd
df_reverted = pd.DataFrame(
    [(h, reverted_commits.get(h, None) is not None) for h in hashes_set],
    columns=["commit_hash", "is_reverted"]
)
# change ai_commits to No_ai_commits to save non-ai
df_reverted.to_csv("csv/ts/No_ai_commits_reverted.csv")

In [ ]:
df_reverted_ai = pd.read_csv("csv/ts/ai_commits_reverted.csv")
df_revertedno_ai = pd.read_csv("csv/ts/No_ai_commits_reverted.csv")

# Count only the reverted commits
count_ai = df_reverted_ai["is_reverted"].sum()         # True counts as 1
count_no_ai = df_revertedno_ai["is_reverted"].sum()

# Bar plot
plt.figure(figsize=(6,6))
plt.bar(["AI Commits", "Non-AI Commits"], [count_ai, count_no_ai], color=["blue", "orange"])

plt.ylabel("Number of Reverted Commits")
plt.title("Reverted Commits: AI vs Non-AI")
plt.show()

In [7]:
import pandas as pd
from github import Github, Auth
from dotenv import load_dotenv
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

# Load GitHub token from .env
load_dotenv()
token = os.getenv("GITHUB_TOKEN")
auth = Auth.Token(token)
g = Github(auth=auth)

# Load dataset
df_commits = pd.read_csv(AI_DATA)
df_commits["failed_pipeline"] = False  # default column

# Connect to repo
repo = g.get_repo("pancakeswap/pancake-frontend")

# Function to check if a commit failed any workflow
def check_failed(commit_sha):
    try:
        runs = repo.get_workflow_runs(head_sha=commit_sha)
        failed = False
        for run in runs:  # iterate properly
            if run.conclusion == "failure":
                failed = True
                break
        return commit_sha, failed
    except Exception as e:
        print(f"Error checking commit {commit_sha}: {e}")
        return commit_sha, False
    
# Parallel execution
results = []
with ThreadPoolExecutor(max_workers=10) as executor:  # adjust max_workers as needed
    future_to_sha = {executor.submit(check_failed, sha): sha for sha in df_commits["hash"]}
    for future in as_completed(future_to_sha):
        sha, failed = future.result()
        results.append((sha, failed))

# Update DataFrame
df_results = pd.DataFrame(results, columns=["hash", "failed_pipeline"])
df_commits = df_commits.merge(df_results, on="hash", how="left")

# Save
df_commits.to_csv("commits_with_pipeline_status.csv", index=False)
print("Saved updated dataset with 'failed_pipeline' column.")

Saved updated dataset with 'failed_pipeline' column.
